In [ ]:
# 1) Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%matplotlib inline

In [ ]:
%cd /content/drive/MyDrive/

In [ ]:
!ls

pyproject.toml	README.md  scripts  solps_ai  solps_ai.egg-info


In [ ]:
![ -d SOLPEx ] || git clone https://github.com/abdoudiaw/SOLPEx.git SOLPEx

In [ ]:
%cd SOLPEx
!git pull origin main

In [ ]:
!pip install -e .

Obtaining file:///content/drive/MyDrive/SOLARIS
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for solps-ai (pyproject.toml) ... done
  Created wheel for solps-ai: filename=solps_ai-0.1.0-0.editable-py3-none-any.whl size=2722 sha256=53aef786f66c6030c660dbf6ca4ec89f54854fd6b0f2bd70a8d9fb3ff6c90c5f
  Stored in directory: /tmp/pip-ephem-wheel-cache-8lasnswp/wheels/b0/27/73/1b5aa94e88a8b655143edd7f62aaf46ac463432771fe918924
Successfully built solps-ai


In [ ]:
# \!grep -R --line-number -E "(^|[^.])import solpex|from solpex" /content/drive/MyDrive/SOLPEx/solpex


/content/drive/MyDrive/Unet2/solps_ai/utils.py:3:from solps_ai.models import UNet, bottleneck_to_z, ParamToZ
/content/drive/MyDrive/Unet2/solps_ai/utils.py:72:from solps_ai.models import bottleneck_to_z


In [ ]:
%cd /content/drive/MyDrive/SOLPEx/scripts/

/content/drive/MyDrive


In [ ]:
!pwd

/content/drive/MyDrive/scripts


In [ ]:
!chmod +x run_full_workflow.sh

In [ ]:
!ls

data					plot_paper_evaluation.py
eval_inverse_cycle_conditional_unet.py	plot_surrogate_comparison_grid.py
eval_source_from_plasma_mesh.py		run_conditional_unet_pipeline.py
get_plasma_data.py			run_full_workflow.sh
make_workflow_figure.py			run_latent_pipeline.py
outputs					run_source_from_plasma_pipeline.py
plot_paper_evaluation_mesh.py


In [ ]:
!pwd

/content/drive/MyDrive


In [ ]:
!NPZ_PATH=data/solps.npz BASE_DIR="/content/drive/MyDrive/SOLPS_DB" ./run_full_workflow.sh

=== Workflow configuration ===
NPZ_PATH=data/solps.npz
BASE_DIR=/content/drive/MyDrive/SOLPS_DB
Y_KEYS_FWD=Te,Ti,ne,ni,ua,Sp,Qe,Qi,Sm
FWD_CHANNEL_WEIGHTS=Te:1.0,Ti:1.0,ne:1.2,ni:1.2,ua:1.5,Sp:1.2,Qe:1.2,Qi:1.2,Sm:1.2
SWEEP_TRIALS=base=32,lr=3e-4,batch=4; base=48,lr=2e-4,batch=4; base=48,lr=1e-4,batch=4
EPOCHS_FWD=450 EARLY_STOP_PATIENCE_FWD=80
INV_N_CASES=40 INV_STEPS=1200
INV_LR=1e-2 INV_N_RESTARTS=5 INV_INIT=noisy_true INV_NOISE_STD=0.2
INV_FIELDS=Te,Ti,ne,ni,ua,Sp,Qe,Qi,Sm
SRC_IN_KEYS=Te,Ti,ne,ni,ua
SRC_OUT_KEYS=Sp,Qe,Qi,Sm
SRC_INCLUDE_PARAMS=1
EPOCHS_SRC=200 EARLY_STOP_PATIENCE_SRC=30
SRC_BASE=32 SRC_LR=3e-4 SRC_BATCH=4
=== Step 1/5: Train params->(plasma+sources) conditional U-Net ===
Device: cuda
cfg: epochs=450 batch=8 base=16 lr=1.00e-03 sweep=True early_stop_patience=80 y_keys=['Te', 'Ti', 'ne', 'ni', 'ua', 'Sp', 'Qe', 'Qi', 'Sm'] sweep_preset=strong3
channel_weights={'Te': 1.0, 'Ti': 1.0, 'ne': 1.2, 'ni': 1.2, 'ua': 1.5, 'Sp': 1.2, 'Qe': 1.2, 'Qi': 1.2, 'Sm': 1.2}
[sweep] tri

**Forward model: inputs scalars-> plasma/sources**

In [ ]:
# !SWEEP=1 EPOCHS=300 EARLY_STOP_PATIENCE=40 \
#   SWEEP_TRIALS="base=32,lr=3e-4,batch=4; base=32,lr=1e-4,batch=4; base=24,lr=2e-4,batch=8" \
#   python run_conditional_unet_pipeline.py

In [ ]:
# !pip install -e /content/drive/MyDrive/quixote-master/

# Plotting

In [ ]:
# !python plot_paper_evaluation_mesh.py \
#     --npz data/solps_native_all_qc.npz \
#     --ckpt outputs/cond_unet.pt \
#     --base-dir "/content/drive/MyDrive/SOLPS_DB" \
#     --all-fields \
#     --error-mode abs \
#     --log-display auto \
#     --outdir outputs/paper_eval_mesh_all_abs

In [ ]:
# !python plot_paper_evaluation_mesh.py \
#     --npz data/solps_native_all_qc.npz \
#     --ckpt outputs/cond_unet.pt \
#     --base-dir "/content/drive/MyDrive/SOLPS_DB" \
#     --all-fields \
#     --paper-grid \
#     --paper-grid-k 0 \
#     --error-mode abs \
#     --error-sign absolute \
#     --log-display auto \
#     --outdir outputs/paper_eval_mesh_all_abs

**Inverse evaluation**

In [ ]:
!python eval_inverse_cycle_conditional_unet.py \
    --npz data/solps_native_all_qc.npz \
    --ckpt outputs/cond_unet.pt \
    --n-cases 3 \
    --steps 400 \
    --fields Te,Ti,ne,ni,ua,Sp,Qe,Qi,Sm \
    --out-csv outputs/inverse_cycle_metrics.csv \
    --out-plot outputs/inverse_param_correlation.png \
    --out-param-corr-csv outputs/inverse_param_correlation.csv

In [ ]:
from IPython.display import Image, display
display(Image(filename="outputs/inverse_param_correlation.png"))

Forward plasma -> sources

In [ ]:
# !python eval_source_from_plasma_mesh.py \
#     --npz data/solps_native_all_qc.npz \
#     --ckpt outputs/source_from_plasma.pt \
#     --base-dir "/content/drive/MyDrive/SOLPS_DB" \
#     --all-fields \
#     --error-mode abs \
#     --error-sign absolute \
#     --log-display auto \
#     --paper-grid \
#     --outdir outputs/source_eval_mesh_abs

In [ ]:
!pwd